<a href="https://colab.research.google.com/github/Michel1412/Aulas_de_IA/blob/main/Trabalho_michas_KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implementação do Knn (Cripe)

In [14]:
import pandas as pd

# URL dos dados
url = "https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/export?format=csv"

# Carregar dados
df = pd.read_csv(url)

# Mapeamento para nomes simples e sem espaços
mapping = {
    "Carimbo de data/hora": "timestamp",
    "Você ficou gripado no ano passado ?": "gripe_ano_passado",
    "Você tomou vacina da gripe no ano passado?": "vacina",
    "  Você frequentou no ano passado,  semanalmente ambientes com muitas pessoas? (salas cheias, ônibus, eventos, etc.)  ": "ambientes_cheios",
    "  Você viajou no ano passado mais de 100 km de distância?  ": "viajou",
    "  Você tem alergia nas vias aéreas (rinite, sinusite, etc.)?  ": "alergia",
    "Quantas horas você dormiu em média por noite no ano passado?": "horas_sono",
    "Você praticou atividade física no ano passado?": "exercicio",
    "Você se alimentou de forma balanceada no ano passado?": "alimentacao",
    "Em média, quantas vezes você lavou as mãos por dia no ano passado?": "lavagem_maos",
    "Na sua percepção, o seu nível de estresse no ano passado foi:": "estresse"
}

# Renomear e limpar
df_cleaned = df.rename(columns=mapping).dropna()

print("Dados carregados e colunas renomeadas:")
display(df_cleaned.head())

Dados carregados e colunas renomeadas:


,timestamp,gripe_ano_passado,vacina,ambientes_cheios,viajou,alergia,horas_sono,exercicio,alimentacao,lavagem_maos,estresse
0,24/03/2026 15:01:35,Sim,Sim,Sim,Poucas vezes por ano,Médio,4 horas ou menos,Sim,Às vezes,3 a 5 vezes,5.0
1,24/03/2026 15:04:20,Sim,Sim,Sim,Nuca,Não,entre 4 e 6 horas,Não,"Não, raramente",Mais de 10 vezes,3.0
2,24/03/2026 15:04:20,Sim,Não,Sim,Poucas vezes por ano,Pouco,mais de 6 horas,Sim,Às vezes,6 a 10 vezes,3.0
3,24/03/2026 15:04:37,Sim,Não,Não,Nuca,Muito,mais de 6 horas,Sim,Às vezes,2 vezes ou menos,2.0
4,24/03/2026 15:05:27,Sim,Sim,Sim,Pelo menos uma vez por mês,Médio,entre 4 e 6 horas,Não,Às vezes,6 a 10 vezes,4.0


In [43]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Criar uma cópia do DataFrame para não modificar o original
df_gripe_processed = df_cleaned.copy()

# --- 1. Codificação de variáveis categóricas binárias (Sim/Não) para 1/0 ---
binary_cols = [
    "gripe_ano_passado",
    "vacina",
    "ambientes_cheios",
    "viajou",
    "alergia",
    "exercicio",
    "alimentacao"
]

for col in binary_cols:
    df_gripe_processed[col] = df_gripe_processed[col].map({"Sim": 1, "Não": 0})

# --- 2. Codificação da variável categórica ordinal 'estresse' ---
# Mapeamento manual para 'estresse' (assumindo ordem ordinal)
estresse_mapping = {
    "Muito baixo": 0,
    "Baixo": 1,
    "Médio": 2,
    "Alto": 3,
    "Muito alto": 4
}
df_gripe_processed["estresse"] = df_gripe_processed["estresse"].map(estresse_mapping)

# --- Mapeamento da coluna 'horas_sono' para valores numéricos ---
horas_sono_mapping = {
    "4 horas ou menos": 4,
    "5 horas": 5,
    "6 horas": 6,
    "7 horas": 7,
    "8 horas": 8,
    "9 horas": 9,
    "10 horas ou mais": 10
}
df_gripe_processed["horas_sono"] = df_gripe_processed["horas_sono"].map(horas_sono_mapping)

# --- Mapeamento da coluna 'lavagem_maos' para valores numéricos ---
lavagem_maos_mapping = {
    "1 a 2 vezes": 1.5, # Usando o ponto médio
    "3 a 5 vezes": 4,   # Usando o ponto médio
    "6 a 9 vezes": 7.5, # Usando o ponto médio
    "10 vezes ou mais": 10
}
df_gripe_processed["lavagem_maos"] = df_gripe_processed["lavagem_maos"].map(lavagem_maos_mapping)


# --- 3. Identificar features numéricas para escalonamento ---
numerical_cols = [
    "horas_sono",
    "lavagem_maos"
]

# --- 4. Escalonamento das features numéricas usando StandardScaler ---
scaler = StandardScaler()
df_gripe_processed[numerical_cols] = scaler.fit_transform(df_gripe_processed[numerical_cols])

# Remover a coluna 'timestamp' que não é uma feature para o modelo
df_gripe_processed = df_gripe_processed.drop(columns=["timestamp"])

print("DataFrame de gripe processado e normalizado:")
display(df_gripe_processed.head())

print("Tipos de dados após o processamento:")
display(df_gripe_processed.info())

DataFrame de gripe processado e normalizado:


,gripe_ano_passado,vacina,ambientes_cheios,viajou,alergia,horas_sono,exercicio,alimentacao,lavagem_maos,estresse
0,1,1,1,NaN,NaN,0.0,1,NaN,0.0,NaN
1,1,1,1,NaN,0.0,NaN,0,NaN,NaN,NaN
2,1,0,1,NaN,NaN,NaN,1,NaN,NaN,NaN
3,1,0,0,NaN,NaN,NaN,1,NaN,NaN,NaN
4,1,1,1,NaN,NaN,NaN,0,NaN,NaN,NaN


Tipos de dados após o processamento:
<class 'pandas.core.frame.DataFrame'>
Index: 185 entries, 0 to 185
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gripe_ano_passado  185 non-null    int64  
 1   vacina             185 non-null    int64  
 2   ambientes_cheios   185 non-null    int64  
 3   viajou             0 non-null      float64
 4   alergia            69 non-null     float64
 5   horas_sono         4 non-null      float64
 6   exercicio          185 non-null    int64  
 7   alimentacao        0 non-null      float64
 8   lavagem_maos       68 non-null     float64
 9   estresse           0 non-null      float64
dtypes: float64(6), int64(4)
memory usage: 15.9 KB


None

In [62]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE

# Criar uma cópia do DataFrame processado
df_model_gripe = df_gripe_processed.copy()

# Preencher todos os NaNs com 0, conforme solicitado pelo usuário
print("Número de NaNs antes de preencher:")
print(df_model_gripe.isnull().sum()[df_model_gripe.isnull().sum() > 0])
df_model_gripe = df_model_gripe.fillna(0)
print("\nNúmero de NaNs após preencher com 0:")
print(df_model_gripe.isnull().sum()[df_model_gripe.isnull().sum() > 0])


# Definir as features que você deseja usar para o treinamento
# Você pode comentar/descomentar as colunas para ativá-las/desativá-las
FEATURES_TO_USE = [
    "vacina",
    "ambientes_cheios",
    "viajou",
    "alergia",
    "horas_sono",
    "exercicio",
    "alimentacao",
    "lavagem_maos",
    "estresse"
]

# Definir Features (X) e Target (y)
X_gripe = df_model_gripe[FEATURES_TO_USE]
y_gripe = df_model_gripe["gripe_ano_passado"]

# Dividir os dados em conjuntos de treino e teste (80% treino, 20% teste)
X_train_gripe, X_test_gripe, y_train_gripe, y_test_gripe = train_test_split(
    X_gripe, y_gripe, test_size=0.2, random_state=42, stratify=y_gripe
)

print(f"\nDimensões do conjunto de treino (X_train_gripe): {X_train_gripe.shape}")
print(f"Dimensões do conjunto de teste (X_test_gripe): {X_test_gripe.shape}")

# Aplicar SMOTE para balancear a base de dados de treino
smote = SMOTE(random_state=42)
X_train_gripe_balanced, y_train_gripe_balanced = smote.fit_resample(X_train_gripe, y_train_gripe)

print(f"\nDimensões do conjunto de treino balanceado (X_train_gripe_balanced): {X_train_gripe_balanced.shape}")
print(f"Distribuição da classe no treino original: {y_train_gripe.value_counts()}")
print(f"Distribuição da classe no treino balanceado: {y_train_gripe_balanced.value_counts()}")

# Treinar o modelo KNN com a base balanceada
knn_n = 5 # Definindo o número de vizinhos para o KNN
knn_gripe_model = KNeighborsClassifier(n_neighbors=knn_n)
knn_gripe_model.fit(X_train_gripe_balanced, y_train_gripe_balanced)

# Avaliar o modelo
y_pred_gripe = knn_gripe_model.predict(X_test_gripe)

print("\n--- Avaliação do Modelo KNN para Gripe (com SMOTE) ---")
print(f"Acurácia: {accuracy_score(y_test_gripe, y_pred_gripe):.2f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test_gripe, y_pred_gripe))
print("\nMatriz de Confusão:\n", confusion_matrix(y_test_gripe, y_pred_gripe))

Número de NaNs antes de preencher:
viajou          185
alergia         116
horas_sono      181
alimentacao     185
lavagem_maos    117
estresse        185
dtype: int64

Número de NaNs após preencher com 0:
Series([], dtype: int64)

Dimensões do conjunto de treino (X_train_gripe): (148, 9)
Dimensões do conjunto de teste (X_test_gripe): (37, 9)

Dimensões do conjunto de treino balanceado (X_train_gripe_balanced): (172, 9)
Distribuição da classe no treino original: gripe_ano_passado
1    86
0    62
Name: count, dtype: int64
Distribuição da classe no treino balanceado: gripe_ano_passado
1    86
0    86
Name: count, dtype: int64

--- Avaliação do Modelo KNN para Gripe (com SMOTE) ---
Acurácia: 0.70

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.64      0.60      0.62        15
           1       0.74      0.77      0.76        22

    accuracy                           0.70        37
   macro avg       0.69      0.69      0.69       

In [63]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# Ensure df_model_gripe is available and processed (from previous cells)
# Assuming X_gripe and y_gripe are already defined from the full dataset
# For individual feature relevance, we need to re-extract X from df_model_gripe

results = []

print("Calculando a acurácia para cada feature individualmente...")

for feature in FEATURES_TO_USE:
    # Define X and y for the current feature
    X_single_feature = df_model_gripe[[feature]]
    y_gripe_single = df_model_gripe["gripe_ano_passado"]

    # Split data for the single feature
    X_train_sf, X_test_sf, y_train_sf, y_test_sf = train_test_split(
        X_single_feature, y_gripe_single, test_size=0.2, random_state=42, stratify=y_gripe_single
    )

    # Train KNN model with the single feature
    knn_sf_model = KNeighborsClassifier(n_neighbors=knn_n)
    knn_sf_model.fit(X_train_sf, y_train_sf)

    # Evaluate the model
    y_pred_sf = knn_sf_model.predict(X_test_sf)
    accuracy_sf = accuracy_score(y_test_sf, y_pred_sf)

    results.append({"Feature": feature, "Accuracy": accuracy_sf})

# Create a DataFrame from the results
relevance_df = pd.DataFrame(results)
relevance_df = relevance_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print("\nRelevância das Features (Acurácia individual):")
display(relevance_df)

Calculando a acurácia para cada feature individualmente...

Relevância das Features (Acurácia individual):


,Feature,Accuracy
0,ambientes_cheios,0.702703
1,exercicio,0.594595
2,vacina,0.405405
3,alergia,0.405405
4,viajou,0.405405
5,horas_sono,0.405405
6,alimentacao,0.405405
7,lavagem_maos,0.405405
8,estresse,0.405405


In [64]:
from sklearn.model_selection import GridSearchCV

# Define the range of n_neighbors to test
param_grid = {'n_neighbors': range(1, 21)}  # Test k from 1 to 20

# Initialize KNN classifier
knn = KNeighborsClassifier()

# Initialize GridSearchCV
# We use the balanced training data (X_train_gripe_balanced, y_train_gripe_balanced)
# StratifiedKFold is suitable for classification to maintain class proportions in folds
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_gripe_balanced, y_train_gripe_balanced)

print(f"Melhor número de vizinhos (knn_n): {grid_search.best_params_['n_neighbors']}")
print(f"Melhor acurácia (com validação cruzada): {grid_search.best_score_:.2f}")

# Optional: Retrain the model with the best k and evaluate on the test set
best_knn_n = grid_search.best_params_['n_neighbors']
best_knn_model = KNeighborsClassifier(n_neighbors=best_knn_n)
best_knn_model.fit(X_train_gripe_balanced, y_train_gripe_balanced)
y_pred_best_knn = best_knn_model.predict(X_test_gripe)

print(f"\nAcurácia no conjunto de teste com melhor knn_n ({best_knn_n}): {accuracy_score(y_test_gripe, y_pred_best_knn):.2f}")
print("\nRelatório de Classificação com melhor knn_n:")
print(classification_report(y_test_gripe, y_pred_best_knn))

Melhor número de vizinhos (knn_n): 15
Melhor acurácia (com validação cruzada): 0.54

Acurácia no conjunto de teste com melhor knn_n (15): 0.38

Relatório de Classificação com melhor knn_n:
              precision    recall  f1-score   support

           0       0.32      0.47      0.38        15
           1       0.47      0.32      0.38        22

    accuracy                           0.38        37
   macro avg       0.39      0.39      0.38        37
weighted avg       0.41      0.38      0.38        37



# Teste do inicio das aulas
Foi usado uma base de dados do chatgpt para classificar o grau da obesidade

In [3]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SHEET_ID = "1kz9n1PQ4k21AONZltkemjcaDDF7NJeC30mfxdZWylIo"
GID = "1894066310"  # aba Página1
url_tsv = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=tsv&gid={GID}"

# TSV + vírgula decimal
df = pd.read_csv(url_tsv, sep="\t", decimal=".")

print("Colunas lidas:", list(df.columns))
display(df.head())

Colunas lidas: ['Nome', 'Idade', 'Altura_m', 'Sexo', 'prc_MassaMagra', 'prc_MassaGorda', 'Peso_kg']


,Nome,Idade,Altura_m,Sexo,prc_MassaMagra,prc_MassaGorda,Peso_kg
0,Lucas Almeida,28,1.72,M,82,18,65
1,Mariana Souza,34,1.65,F,74,26,60
2,Pedro Santos,22,1.80,M,85,15,72
3,Ana Clara Lima,29,1.60,F,75,25,56
4,Rafael Costa,31,1.75,M,80,20,68


In [4]:
FEATURES = ["Idade", "Altura_m", "prc_MassaMagra", "prc_MassaGorda"]
FEATURES_STRING = ["Nome"]
FEATURES_CLASSES = ["Sexo"]
FEATURES_NUMERO = ["Idade", "Altura_m", "prc_MassaMagra", "prc_MassaGorda", "Peso_kg"]


TARGET = "Peso_kg"            # coluna F (classe)

for col in FEATURES :
    df[col] = pd.to_numeric(df[col], errors="coerce")

# for coln in FEATURES_STRING :
#     df[coln] = df[coln].astype(str).str.strip().str.lower()

# for col in FEATURES_CLASSES :
#     df[col] = df[col].astype(str).str.strip().str.lower()

# Normaliza o target e remove linhas inválidas

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=FEATURES + [TARGET]).copy() #remove linhas com valores inválidos

print("\nDistribuição do Nome:")
print(df[FEATURES_STRING].value_counts())



Distribuição do Nome:
Nome               
Alan Mendes            1
Alessandra Carvalho    1
Alexandre Silva        1
Aline Batista          1
Amanda Lopes           1
                      ..
Vanessa Borges         1
Vinicius Duarte        1
Vitor Hugo             1
Wesley Fonseca         1
Yasmin Freitas         1
Name: count, Length: 92, dtype: int64


In [5]:
le = LabelEncoder()
y_all = le.fit_transform(df[TARGET])
X_all = df[FEATURES].values

print("\nClasses:", list(le.classes_))
print("Total linhas válidas:", len(df))


Classes: [np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(67), np.int64(68), np.int64(69), np.int64(70), np.int64(71), np.int64(72), np.int64(73), np.int64(74), np.int64(75), np.int64(76), np.int64(79)]
Total linhas válidas: 92


In [6]:

RANDOM_STATE = 1412
df_shuffled = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
limite_teste=int(len(df)*0.20) # Split : 20% teste + 80% treino
test_df  = df_shuffled.iloc[:limite_teste].copy()
train_df = df_shuffled.iloc[limite_teste:len(df)].copy()
X_train = train_df[FEATURES].values
y_train = le.transform(train_df[TARGET])
X_test  = test_df[FEATURES].values
y_test  = le.transform(test_df[TARGET])
print("\nTreino:", len(train_df), "| Teste:", len(test_df))



Treino: 74 | Teste: 18


In [7]:
def dist_classes(frame, target_col, titulo):
    vc = frame[target_col].value_counts(dropna=False)
    pct = (vc / len(frame) * 100).round(1)
    out = pd.DataFrame({"qtd": vc, "%": pct})
    print(f"\n=== {titulo} (n={len(frame)}) ===")
    display(out)

dist_classes(df, TARGET, "Dataset completo")
dist_classes(train_df, TARGET, "Treino")
dist_classes(test_df, TARGET, "Teste")


=== Dataset completo (n=92) ===


,qtd,%
Peso_kg,,
60,6,6.5
61,6,6.5
58,6,6.5
65,5,5.4
74,5,5.4
63,5,5.4
62,5,5.4
68,5,5.4
66,4,4.3



=== Treino (n=74) ===


,qtd,%
Peso_kg,,
60,5,6.8
65,5,6.8
68,5,6.8
74,5,6.8
62,5,6.8
58,5,6.8
72,4,5.4
70,4,5.4
56,4,5.4



=== Teste (n=18) ===


,qtd,%
Peso_kg,,
63,3,16.7
61,2,11.1
66,2,11.1
57,2,11.1
64,2,11.1
67,2,11.1
58,1,5.6
60,1,5.6
54,1,5.6


In [8]:
# Modelo KNN
K = 5
WEIGHTS = "distance"  # "uniform" ou "distance"

model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=K, weights=WEIGHTS))
])

model.fit(X_train, y_train)


Pipeline(steps=[('scaler', StandardScaler()),
                ('knn', KNeighborsClassifier(weights='distance'))])

In [10]:
import numpy as np

# Avaliação

y_pred = model.predict(X_test)

print("\nFeatures usadas:", FEATURES)
print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nRelatório:")

# Get the unique labels that actually appear in y_test and y_pred
# This gives the integer-encoded labels
actual_labels_in_test_set = np.unique(np.concatenate((y_test, y_pred)))

# Get the corresponding original class names (Peso_kg values) for these labels
# le.classes_ maps directly from integer index to the original value
actual_target_names = [str(le.classes_[label_idx]) for label_idx in actual_labels_in_test_set]

# Now, call classification_report with the filtered labels and target names
print(classification_report(y_test, y_pred, labels=actual_labels_in_test_set, target_names=actual_target_names))
print("Matriz de confusão (real x previsto):")
print(confusion_matrix(y_test, y_pred))


Features usadas: ['Idade', 'Altura_m', 'prc_MassaMagra', 'prc_MassaGorda']
Acurácia: 0.1111111111111111

Relatório:
              precision    recall  f1-score   support

          54       0.00      0.00      0.00         1
          56       0.00      0.00      0.00         0
          57       0.00      0.00      0.00         2
          58       0.50      1.00      0.67         1
          60       1.00      1.00      1.00         1
          61       0.00      0.00      0.00         2
          62       0.00      0.00      0.00         0
          63       0.00      0.00      0.00         3
          64       0.00      0.00      0.00         2
          65       0.00      0.00      0.00         0
          66       0.00      0.00      0.00         2
          67       0.00      0.00      0.00         2
          68       0.00      0.00      0.00         0
          71       0.00      0.00      0.00         1
          73       0.00      0.00      0.00         1
          74      

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [12]:
# Resultado detalhado do teste
test_df["classe_prevista"] = le.inverse_transform(y_pred)
display(test_df[FEATURES + [TARGET, "classe_prevista"]])

,Idade,Altura_m,prc_MassaMagra,prc_MassaGorda,Peso_kg,classe_prevista
0,26,1.67,73,27,61,62
1,33,1.73,80,20,66,68
2,28,1.70,74,26,63,62
3,24,1.64,75,25,58,58
4,32,1.58,76,24,54,56
5,28,1.62,74,26,57,58
6,33,1.65,73,27,60,60
7,27,1.73,82,18,66,65
8,34,1.62,71,29,57,61
9,36,1.70,79,21,64,68
